국토교통부데이터 불러오기
=

In [1]:
import pandas as pd
import numpy as np

In [2]:
df2018=pd.read_csv('raw/서울2018.csv', index_col=0)
df2019=pd.read_csv('raw/서울2019.csv', index_col=0)
df2020=pd.read_csv('raw/서울2020.csv', index_col=0)
df2021=pd.read_csv('raw/서울2021.csv', index_col=0)
df2022=pd.read_csv('raw/서울2022.csv', index_col=0)
df2023=pd.read_csv('raw/서울2023.csv', index_col=0)

df = pd.concat([df2018,df2019,df2020,df2021,df2022,df2023])

# # 2023년도 3분기 행 제거
df=df[df['계약년월']<202307]

외부데이터 추가 할 때 필요한 전처리
=

In [3]:
def date_str(x):
    if x<10:
        return '0'+str(x)
    else:
        return str(x)

# 년도 / 월 분리 생성
df['계약년도']=df['계약년월'].astype(str).str[0:4].astype(int)
df['계약월']=df['계약년월'].astype(str).str[4:].astype(int)

In [4]:
#번지 특이값 제거
df['본번']=df['본번'].fillna(0)
df['본번'] = df['본번'].replace([np.inf, -np.inf], np.nan).astype(int)

df['부번']=df['부번'].fillna(0)
df['부번'] = df['부번'].replace([np.inf, -np.inf], np.nan).astype(int)

df['번지'] = df.apply(lambda row: str(row['본번']) if row['부번'] == 0 else f"{row['본번']}-{row['부번']}", axis=1)

In [5]:
# 시군구를 3개로 나눔
df[['시','구','동']]=df['시군구'].str.split(expand=True)

In [6]:
# 부동산데이터와 번지, 도로명을 기준으로 합치기 위해 도로명 형식 변경
df['도로명'] = df['시']+' '+df['구']+' '+df['도로명']

In [7]:
df=df[['구', '동','번지','도로명','전용면적(㎡)','계약년도','계약월', '계약일','거래금액(만원)','층','건축년도']]

평당가격, 건축경과(년) 추가
=

In [8]:
df['전용면적'] = round(df['전용면적(㎡)']*0.3025, 2)
df.drop(columns='전용면적(㎡)', inplace=True)

# 평당가격
df['거래금액(만원)']=df['거래금액(만원)'].str.replace(',','').astype(int)
df['평당가격'] = round(df['거래금액(만원)']/df['전용면적'])
df.drop(columns='거래금액(만원)', inplace=True)

# 건축경과
df['건축경과'] = df['계약년도'] - df['건축년도']
df.drop(columns='건축년도', inplace=True)

부동산데이터 병합
=

In [9]:
d1=pd.read_csv('raw/서울시 강남구.csv', index_col=0)
d2=pd.read_csv('raw/서울시 강동구.csv', index_col=0)
d3=pd.read_csv('raw/서울시 강북구.csv', index_col=0)
d4=pd.read_csv('raw/서울시 강서구.csv', index_col=0)
d5=pd.read_csv('raw/서울시 관악구.csv', index_col=0)
d6=pd.read_csv('raw/서울시 광진구.csv', index_col=0)
d7=pd.read_csv('raw/서울시 구로구.csv', index_col=0)
d8=pd.read_csv('raw/서울시 금천구.csv', index_col=0)
d9=pd.read_csv('raw/서울시 노원구.csv', index_col=0)
d10=pd.read_csv('raw/서울시 도봉구.csv', index_col=0)
d11=pd.read_csv('raw/서울시 동대문구.csv', index_col=0)
d12=pd.read_csv('raw/서울시 동작구.csv', index_col=0)
d13=pd.read_csv('raw/서울시 마포구.csv', index_col=0)
d14=pd.read_csv('raw/서울시 서대문구.csv', index_col=0)
d15=pd.read_csv('raw/서울시 서초구.csv', index_col=0)
d16=pd.read_csv('raw/서울시 성동구.csv', index_col=0)
d17=pd.read_csv('raw/서울시 성북구.csv', index_col=0)
d18=pd.read_csv('raw/서울시 송파구.csv', index_col=0)
d19=pd.read_csv('raw/서울시 양천구.csv', index_col=0)
d20=pd.read_csv('raw/서울시 영등포구.csv', index_col=0)
d21=pd.read_csv('raw/서울시 용산구.csv', index_col=0)
d22=pd.read_csv('raw/서울시 은평구.csv', index_col=0)
d23=pd.read_csv('raw/서울시 종로구.csv', index_col=0)
d24=pd.read_csv('raw/서울시 중구.csv', index_col=0)
d25=pd.read_csv('raw/서울시 중랑구.csv', index_col=0)

d = pd.concat([d1,d2,d3,d4,d5,d6,d7,d8,d9,d10,d11,d12,d13,d14,d15,d16,d17,d18,d19,d20,d21,d22,d23,d24,d25]).drop(columns='Unnamed: 0')

In [10]:
# 이름변경, 필요없는 값 제거
d.rename(columns={'법정동주소':'번지','도로명주소':'도로명'}, inplace=True)
d['도로명']=d['도로명'].str.replace('서울시','서울특별시')
d=d[['번지','도로명','latitude','longitude','세대수','주차대수','건설사','난방']]

# object타입은 첫번째값, 정수, 실수값은 평균값으로 '번지'를 기준으로 Groupby
d = d.groupby(['번지','도로명']).agg({
    '건설사': 'first',
    '난방': 'first',
    'latitude': 'mean',
    'longitude': 'mean',
    '세대수': lambda x: int(x.mean()),
    '주차대수': lambda x: int(x.mean())
}).reset_index()

# 번지를 d - d 형식으로 변경 (한글부분 제거
d['번지']=d['번지'].str.split(expand=True)[3]
d=d[d['번지'].str.contains(r'\d')]

# 번지와 도로명을 기준으로 합병
df = pd.merge(df, d, on=['도로명','번지'],how='left')

df = df.dropna()

세대당 주차대수 추가
=

In [11]:
#주차대수
df['주차대수'] = round(df['주차대수']/df['세대수'], 2)

금리데이터 병합
=

In [12]:
kbank = pd.read_csv('raw/한국은행.csv')

In [13]:
kbank.columns=['년도','월','일','금리']

kbank['날짜'] = kbank['년도'].astype(str)+'-'+kbank['월'].astype(str)+'-'+kbank['일'].astype(str)
df['계약날짜'] = df['계약년도'].astype(str)+'-'+df['계약월'].astype(str)+'-'+df['계약일'].astype(str)

kbank['날짜'] = pd.to_datetime(kbank['날짜'])
df['계약날짜'] = pd.to_datetime(df['계약날짜'])

In [14]:
def get_interest_rate(row):
    matching_row = kbank[kbank['날짜'] <= row] # 해당 날짜 이전의 가장 최근 데이터 선택
    if not matching_row.empty:
        return matching_row.iloc[0]['금리']
    else:
        return None

# apply 함수를 사용하여 각 행에 대해 금리 값을 가져와서 '금리' 칼럼에 저장
df['금리'] = df['계약날짜'].apply(get_interest_rate)

개별공시지가데이터 병합
=

In [15]:
a1=pd.read_csv('raw/공시지가_2018년.csv',encoding = 'cp949', low_memory = False)
a2=pd.read_csv('raw/공시지가_2019년.csv',encoding = 'cp949', low_memory = False)
a3=pd.read_csv('raw/공시지가_2020년.csv',encoding = 'cp949', low_memory = False)
a4=pd.read_csv('raw/공시지가_2021년.csv',encoding = 'cp949', low_memory = False)
a5=pd.read_csv('raw/공시지가_2022년.csv',encoding = 'cp949', low_memory = False)
a6=pd.read_csv('raw/공시지가_2023년.csv',encoding = 'cp949', low_memory = False)

gong = pd.concat([a1,a2,a3,a4,a5,a6])

In [16]:
gong.drop(columns=['시도명','토지코드','시군구코드','법정동코드','필지구분코드','필지구분명','본번','부번','기준년월'],inplace=True)
gong.rename(columns={'공시지가(원/㎡)':'공시지가','시군구명':'구','법정동명':'동'},inplace=True)

gong = gong.groupby(['구','동','기준년도'])['공시지가'].agg('mean').round(2).reset_index()

gong['공시지가'] = round(gong['공시지가']*3.3058/10000, 2)

gong.rename(columns={'기준년도':'계약년도'}, inplace=True)
df = pd.merge(df, gong, on=['구','동','계약년도'], how='left')

혼인10년차데이터 병합
=

In [17]:
ten = pd.read_csv('raw/혼인10년차.csv')

In [18]:
ten = ten[ten['동'] != '소계']
ten.drop(columns='동',inplace=True)

ten = pd.melt(ten, id_vars='구', var_name='계약년도', value_name='혼인10년차')

ten['계약년도']=ten['계약년도'].astype(int)+10
ten['혼인10년차'] = ten['혼인10년차'].str.replace('-','0').astype(int)

ten = ten.groupby(['구','계약년도'])['혼인10년차'].sum().reset_index()

df = pd.merge(df, ten, on=['구','계약년도'], how='left')

달러환율데이터 병합
=

In [19]:
dol = pd.read_csv('raw/환율달러.csv',encoding='cp949')

In [20]:
dol.rename(columns=lambda x: x.strip(), inplace=True)
dol.rename(columns={'종가':'달러'}, inplace=True)

dol[['계약년도','계약월','일']] = dol['날짜'].str.split('-', expand=True).astype(int)
dol=dol.drop(['날짜','시가','고가','저가','일'], axis=1)

df = pd.merge(df, dol, on=['계약년도','계약월'], how='left')

경제성장률(실질GDP성장률) 병합
=

In [21]:
gdp = pd.read_csv('raw/주요지표(분기지표).csv')

In [22]:
# 분기 column 생성
def QQQ(x):
    if x>=1 and x<=3:
        return 1
    elif x>=4 and x<=6:
        return 2
    elif x>=7 and x<=9:
        return 3
    else: return 4

df['분기']=df['계약월'].apply(lambda x: QQQ(x))

In [23]:
gdp.drop(columns=['통계표','계정항목','단위','변환'], inplace=True)

gdp = pd.melt(gdp, var_name='날짜', value_name='GDP성장률')

gdp[['계약년도','분기']]=gdp['날짜'].str.split('/Q', expand=True).astype(int)
gdp.drop(columns='날짜', inplace=True)

df = pd.merge(df, gdp, on=['계약년도','분기'], how='left')

거주인구 병합
=

In [24]:
pp = pd.read_csv('raw/주민등록인구.csv')

In [25]:
pp.drop(columns='동별(1)', inplace=True)
pp=pp.drop(0).drop(1)
pp.rename(columns={'동별(2)':'구'}, inplace=True)

pp = pd.melt(pp, id_vars=['구'], var_name='날짜', value_name='인구수')

pp['날짜']=pp['날짜'].str.replace('/4', '')
pp[['계약년도','분기']]=pp['날짜'].str.split(' ', expand=True).astype(int)
pp.drop(columns='날짜', inplace=True)

df = pd.merge(df, pp, on=['계약년도','분기','구'], how='left')

평균연령 병합
=

In [26]:
old = pd.read_csv('raw/평균연령.csv')

In [27]:
old.drop(columns='동별(1)',inplace=True)
old=old.drop(0)
old.rename(columns={'동별(2)':'구'}, inplace=True)
old.columns = old.columns.str.strip()

old = pd.melt(old, id_vars=['구'], var_name='날짜', value_name='평균연령')

old['날짜']=old['날짜'].str.replace('/4', '')
old[['계약년도','분기']]=old['날짜'].str.split(' ', expand=True).astype(int)
old.drop(columns='날짜', inplace=True)

df = pd.merge(df, old, on=['계약년도','분기','구'], how='left')

In [28]:
df.to_csv('전체병합.csv')

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 272780 entries, 0 to 272779
Data columns (total 27 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   구          272780 non-null  object        
 1   동          272780 non-null  object        
 2   번지         272780 non-null  object        
 3   도로명        272780 non-null  object        
 4   계약년도       272780 non-null  int64         
 5   계약월        272780 non-null  int64         
 6   계약일        272780 non-null  int64         
 7   층          272780 non-null  int64         
 8   전용면적       272780 non-null  float64       
 9   평당가격       272780 non-null  float64       
 10  건축경과       272780 non-null  float64       
 11  건설사        272780 non-null  object        
 12  난방         272780 non-null  object        
 13  latitude   272780 non-null  float64       
 14  longitude  272780 non-null  float64       
 15  세대수        272780 non-null  float64       
 16  주차대수       272780 no